# 이메일 분류 AI 플랫폼

In [18]:
import numpy as np
import pandas as pd
import re

# DataLoad

In [2]:
df = pd.read_csv('../data/email_data.csv', parse_dates=["received_at"])

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12201 entries, 0 to 12200
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   email_id        12201 non-null  int64         
 1   thread_id       12201 non-null  int64         
 2   received_at     12201 non-null  datetime64[ns]
 3   sender          12201 non-null  object        
 4   recipient       12201 non-null  object        
 5   title           12201 non-null  object        
 6   content         12201 non-null  object        
 7   has_attachment  12201 non-null  int64         
 8   mail_channel    12201 non-null  int64         
 9   label_dept      12201 non-null  int64         
 10  priority_level  12201 non-null  int64         
 11  sentiment       12201 non-null  int64         
 12  is_complaint    12201 non-null  int64         
 13  assignee        12201 non-null  int64         
 14  is_spam         12201 non-null  int64         
dtypes:

In [4]:
len(df)

12201

In [5]:
df.head(1)

,email_id,thread_id,received_at,sender,recipient,title,content,has_attachment,mail_channel,label_dept,priority_level,sentiment,is_complaint,assignee,is_spam
0,10001,1,2024-07-10 10:00:00,dev.seojun@ssacorp.com,service.minji@ssacorp.com,API 서버 에러 로그 분석 요청,"안녕하세요, 서비스팀 민지님. 어제부터 API 서버에서 간헐적으로 에러가 발생하고 ...",1,1,0,2,1,0,0,0


## 스키마 검증

### 값 범위 체크

In [8]:
def check_ranges(df: pd.DataFrame):
    assert df["priority_level"].between(0, 3).all(), "priority_level 범위 오류"
    assert df["sentiment"].between(0, 2).all(), "sentiment 범위 오류"
    assert df["is_complaint"].between(0, 3).all(), "is_complaint 범위 오류"
    assert df["assignee"].between(0, 4).all(), "assignee 범위 오류"
    
    # label_dept는 -1(스팸) 또는 0~10
    valid_dept = df["label_dept"].isin([-1] + list(range(0, 11)))
    assert valid_dept.all(), "label_dept 범위 오류"


### 스팸 규칙 체크

In [9]:
def check_spam_rules(df: pd.DataFrame):
    spam = df[df["is_spam"] == 1]
    if not spam.empty:
        assert (spam["mail_channel"] == 0).all(), "스팸인데 mail_channel != 0 존재"
        assert (spam["label_dept"] == -1).all(), "스팸인데 label_dept != -1 존재"


### 도메인 규칙 체크

In [10]:
def check_domains(df: pd.DataFrame):
    # 사내 직원이 받는 메일 → recipient는 항상 @ssacorp.com
    assert df["recipient"].str.endswith("@ssacorp.com").all(), "recipient 중 사내 도메인 아님"
    
    # 스팸은 sender가 일반 도메인인지 대략 체크
    spam = df[df["is_spam"] == 1]
    common_spam_domains = ("gmail.com", "naver.com", "daum.net", "yahoo.com")
    if not spam.empty:
        mask = spam["sender"].str.endswith(common_spam_domains)
        # 100% 일치까지는 안봐도 되니, 비율 정도 확인
        spam_ratio = mask.mean()
        print(f"[INFO] 스팸 중 일반 도메인 비율: {spam_ratio:.2%}")


### 중복된 title, content 체크

In [13]:
def check_duplicates(df: pd.DataFrame):
    # email_id는 고유해야 함
    
    
    # title+content 완전 동일한 행 수
    dup_tc = df.duplicated(subset=["title", "content"]).sum()
    print(f"[INFO] title+content 완전 중복 행 개수: {dup_tc}")

### 체크 실행

In [17]:
def validate_email_schema(df: pd.DataFrame):
    print("[STEP] 값 범위 체크")
    check_ranges(df)
    print("  ✔ OK")
    
    print("[STEP] 스팸 규칙 체크")
    check_spam_rules(df)
    print("  ✔ OK")
    
    print("[STEP] 도메인 규칙 체크")
    check_domains(df)
    
    print("[STEP] 중복 여부 체크")
    check_duplicates(df)
    print("  ✔ Done")


In [15]:
validate_email_schema(df)

[STEP] 값 범위 체크
  ✔ OK
[STEP] 스팸 규칙 체크
  ✔ OK
[STEP] 도메인 규칙 체크 (라이트)
[INFO] 스팸 중 일반 도메인 비율: 100.00%
[STEP] 중복 여부 체크
[INFO] title+content 완전 중복 행 개수: 6
  ✔ Done (중복 있어도 지금은 정보만 확인)


# 전처리 & 가공

## title, content 통합 피처 생성
* 제목과 본문을 붙여서 더 정확한 의미를 이해하게 하기 위해

In [19]:
def build_text_column(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    # 기본 결합
    text = df["title"].astype(str) + "\n" + df["content"].astype(str)
    
    # 연속 공백, 줄바꿈 최소 정리 (너무 과한 정규식은 X)
    text = text.str.replace(r"\s+", " ", regex=True)
    
    df["text"] = text.str.strip()
    return df

df = build_text_column(df)

In [20]:
df['text']

0        API 서버 에러 로그 분석 요청 안녕하세요, 서비스팀 민지님. 어제부터 API 서...
1        신규 캠페인 관련 영업팀 협조 요청 안녕하세요, 영업팀 지훈님. 이번에 진행하는 신...
2        2분기 실적 보고서 검토 요청 안녕하세요, 경영팀 동현님. 2분기 실적 보고서를 첨...
3        개발팀 인사 발령 안내 안녕하세요, 개발팀 서준님. 금번 인사 발령에 따라 개발팀에...
4        신규 UI/UX 디자인 시안 검토 요청 안녕하세요, 디자인팀 지은님. 현재 기획 중...
                               ...                        
12196    세금계산서 발행 관련 협조 요청 강소현님, 안녕하세요. 거래처 솔루텍에서 세금계산서...
12197    신규 UI 디자인 시안 검토 요청 정현우님, 안녕하세요. 신규 UI 디자인 시안이 ...
12198    인사 발령에 대한 불만 사항 접수 박재현님, 안녕하세요. 이번 인사 발령에 대한 불...
12199    서버 장애 발생 및 긴급 복구 요청 박은지님, 안녕하세요. 현재 주 서버에서 장애가...
12200    업무 협조 요청에 대한 지연 한지원님, 안녕하세요. 지난주 요청드린 업무 협조 건에...
Name: text, Length: 12201, dtype: object

## 시간/텍스트 길이 기반 피처 추가
* 긴급도/업무량 패턴분석에 사용
* 직원 성실도 or 업무량 시계열
* 시간별 메일 패턴 분석시 사용

In [21]:
def add_basic_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    # 시간 관련
    df["received_at"] = pd.to_datetime(df["received_at"])
    df["hour"] = df["received_at"].dt.hour
    df["dayofweek"] = df["received_at"].dt.dayofweek  # 0=월, 6=일
    df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)
    
    # 텍스트 길이 관련
    df["text_len_char"] = df["text"].str.len()
    df["text_len_word"] = df["text"].str.split().str.len()
    
    return df

df = add_basic_features(df)

In [24]:
df.head(1)

,email_id,thread_id,received_at,sender,recipient,title,content,has_attachment,mail_channel,label_dept,...,sentiment,is_complaint,assignee,is_spam,text,hour,dayofweek,is_weekend,text_len_char,text_len_word
0,10001,1,2024-07-10 10:00:00,dev.seojun@ssacorp.com,service.minji@ssacorp.com,API 서버 에러 로그 분석 요청,"안녕하세요, 서비스팀 민지님. 어제부터 API 서버에서 간헐적으로 에러가 발생하고 ...",1,1,0,...,1,0,0,0,"API 서버 에러 로그 분석 요청 안녕하세요, 서비스팀 민지님. 어제부터 API 서...",10,2,0,132,29


## 스팸/정상 분리, 학습용 DF

In [26]:
def split_spam_normal(df: pd.DataFrame):
    df_spam = df[df["is_spam"] == 1].copy()
    df_normal = df[df["is_spam"] == 0].copy()
    
    # 정상 메일에서도 label_dept = -1인 건 제거 (이론상 없어야 하지만 방지차원)
    df_normal = df_normal[df_normal["label_dept"] >= 0].copy()
    
    return df_normal, df_spam

df_normal, df_spam = split_spam_normal(df)
print(df_normal.shape, df_spam.shape)


(11331, 21) (868, 21)


In [30]:
df_normal['is_spam'].unique()

array([0])

In [29]:
df_spam['is_spam'].unique()

array([1])

## 첫번째 전처리 파이프라인 한방 코드

In [ ]:
# def preprocess_stage1(path: str | Path):
#     # 1. 로드
#     df_raw = load_email_data(path)
    
#     # 2. 타입 캐스팅
#     df = cast_types(df_raw)
    
#     # 3. 스키마/규칙 검증
#     validate_email_schema(df)
    
#     # 4. 텍스트 컬럼 생성
#     df = build_text_column(df)
    
#     # 5. 기본 피처 추가
#     df = add_basic_features(df)
    
#     # 6. 스팸/정상 분리
#     df_normal, df_spam = split_spam_normal(df)
    
#     return df, df_normal, df_spam

# df_all, df_normal, df_spam = preprocess_stage1(DATA_PATH)

In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12201 entries, 0 to 12200
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   email_id        12201 non-null  int64         
 1   thread_id       12201 non-null  int64         
 2   received_at     12201 non-null  datetime64[ns]
 3   sender          12201 non-null  object        
 4   recipient       12201 non-null  object        
 5   title           12201 non-null  object        
 6   content         12201 non-null  object        
 7   has_attachment  12201 non-null  int64         
 8   mail_channel    12201 non-null  int64         
 9   label_dept      12201 non-null  int64         
 10  priority_level  12201 non-null  int64         
 11  sentiment       12201 non-null  int64         
 12  is_complaint    12201 non-null  int64         
 13  assignee        12201 non-null  int64         
 14  is_spam         12201 non-null  int64         
 15  te